# Udeco 接触状态分解：单样本双通道实验

本 Notebook 用于在一个真实样本上独立检查 Udeco2 与 Udeco3 的接触状态分解。核心问题是：先用临时趋势识别接触模态切换，再重建连续的接触模态信号，最后估计中尺度趋势与条件更新的超慢基线。

物理分解为：

\[
U(t)=B_{\mathrm{slow}}(t)+P(t)+J(t)+n(t)
\]

- `J_events` 只描述 attach / detach 边沿；`J(t)` 描述当前接触模态差，不再累计历史阶跃。
- `P(t)` 表示真实接触状态下的连续压迫 / 松弛分量；Non-contact 中严格为 0。
- `Bslow(t)` 只在真实且稳定的 Contact 点更新，其余状态严格冻结。
- Udeco2 / Udeco3 使用同一函数接口，但噪声、候选、状态和基线全部独立估计。

In [ ]:
# =============================================================================
# 统一实验参数：通常只需修改 SAMPLE_NAME；其余参数用于有针对性地调试
# =============================================================================

SAMPLE_NAME = "cgx_gripper_1"
# 样本名，不含 `_sensor.csv` 后缀。修改后会在 DATA_ROOT 下唯一匹配对应文件。

DATA_ROOT = "subject_data"
# 数据根目录。通常不需要修改；目录内按受试者子目录保存 sensor CSV。

PROTOCOL_FS_HZ = 100.0
# 协议采样率（Hz）。增大/减小不会改变数据本身，只影响秒到采样点的换算；推荐保持采集协议的 100 Hz。

# =========================
# Hampel：轻度去除孤立尖峰
# =========================
HAMPEL_WINDOW_SEC = 0.15
# 局部中位数窗口（秒）。增大可覆盖更宽尖峰但更可能触碰真实边沿；减小更保守但可能漏掉宽脉冲。推荐 0.08–0.25 s。

HAMPEL_N_SIGMA = 3.5
# 离群阈值（局部 MAD 倍数）。增大时只删除更极端尖峰；减小时清理更强但可能误删真实变化。推荐 3.0–5.0。

# =========================
# Bpilot：仅供检测使用的临时趋势
# =========================
PILOT_LOESS_WINDOW_SEC = 3.0
# 临时 Robust LOWESS 的等效窗口（秒）。增大使 Bpilot 更平滑、fast residual 含更多慢变化；减小使 Bpilot 更贴近原信号。推荐 2–5 s。

PILOT_LOESS_ROBUST_ITERS = 2
# LOWESS 稳健重加权次数。增大更抗离群但计算更慢；减小更快但更易被异常拖动。推荐 1–3。

# =========================
# attach / detach 多尺度候选
# =========================
CANDIDATE_SCALES_SEC = (0.05, 0.10, 0.20, 0.40, 0.80, 1.60)
# 检测的过渡持续时间尺度（秒）。增加长尺度可发现更慢 attach 但会混入压力变化；删除长尺度更保守。推荐覆盖 0.05–1.6 s。

CANDIDATE_Z_THRESHOLD = 4.0
# 前后平台差相对局部 MAD 的阈值。增大减少噪声误报但可能漏检小阶跃；减小提高灵敏度但候选增多。推荐 3.5–7.0。

CANDIDATE_MERGE_SEC = 0.25
# 同方向相邻候选的合并间隔（秒）。增大使多个局部响应合成一个事件；减小会保留更多细分候选。推荐 0.10–0.40 s。

REFINEMENT_CONTEXT_SEC = 1.5
# 候选精修时观察前后平台的上下文（秒）。增大增强平台统计稳定性但可能跨越其他事件；减小更局部但更受噪声影响。推荐 0.8–2.5 s。

CANDIDATE_REFINE_SMOOTH_SEC = 0.20
# 10%–90% 边界精修前的短中值平滑窗口（秒）。增大可避免噪声把慢 attach 压成单点；减小会保留更陡细节。推荐 0.10–0.35 s。

DETACH_PEAK_SLOPE_Z = 3.0
# 快速 detach 的峰值斜率显著性阈值。增大只接受更陡峭的脱离；减小会把快速压力变化也当作 detach。推荐 3–8。

ATTACH_MIN_MONOTONICITY = 0.70
# 慢 attach 的整体单调性下限（0–1）。增大要求波形更连续单调；减小允许更多抖动也更易误判。推荐 0.60–0.85。

# =========================
# 保守有限状态机
# =========================
INITIAL_CONTACT_SEC = 30.0
# 开始阶段强制 Contact 的时长（秒），用于建立噪声/斜率参考。增大参考更稳但会屏蔽早期真实切换；减小更早允许切换。推荐 20–40 s。

STATE_MIN_DWELL_SEC = 0.5
# 确认一个 Non-contact 区间所需的最短驻留（秒）。增大可拒绝短暂扰动；减小可识别很短脱离。推荐 0.3–1.0 s。

STATE_TRANSITION_CONFIDENCE = 0.50
# 状态机接受候选的最低综合置信度（0–1）。增大更保守、漏检可能增加；减小接受更多边沿。推荐 0.45–0.75。

CONTACT_ANCHOR_WINDOW_SEC = 1.0
# Ucontact 插值两侧真实接触锚窗（秒）。增大使锚值更稳但可能平均掉压力趋势；减小更局部但易受噪声影响。推荐 0.5–2.0 s。

MAX_DYNAMIC_FRACTION = 0.15
# 非接触候选区允许的高 Bpilot 斜率样本比例。增大更容易确认 Non-contact；减小更保守地保留压力活动。推荐 0.05–0.25。

STATE_PILOT_SLOPE_RELATIVE_THRESHOLD = 0.03
# 将 Bpilot 斜率判为“中尺度活动”的相对阈值（每秒相对初始接触水平）。增大时压力活动更少触发冻结；减小时状态机更保守。推荐 0.02–0.05。

CONTACT_ANCHOR_RESIDUAL_FRACTION = 0.07
# 接触锚窗允许的 clean-Bpilot 稳健噪声占初始接触水平的比例。增大允许更粗糙的真实接触锚点；减小只接受更平稳锚点。推荐 0.04–0.10。

TERMINAL_MIN_TAIL_SEC = 2.0
# 仅有末尾 detach 时，后续稳定尾段的最短长度（秒）。增大减少误判；减小允许更靠近结尾的脱离。推荐 1–5 s。

DOWNSTREAM_MIN_SEGMENT_SEC = 0.25
# 下游稳健处理所用的最短片段尺度（秒）。增大抑制短时噪声；减小保留更短结构。推荐 0.10–0.50 s。

# =========================
# 最终 Bm_contact：接触模态中尺度趋势
# =========================
CONTACT_LOESS_WINDOW_SEC = 3.0
# Ucontact 上最终 Robust LOWESS 的等效窗口（秒）。增大更平滑但会削弱快速压迫坡；减小保留细节但噪声更多。推荐 1.5–6 s。

CONTACT_LOESS_ROBUST_ITERS = 2
# 最终 LOWESS 的稳健迭代次数。增大更抗少量残余异常；减小计算更快。推荐 1–3。

# =========================
# Bslow：仅稳定接触时更新的超慢基线
# =========================
BSLOW_TAU_SEC = 60.0
# 慢基线时间常数（秒）。增大使 Bslow 更慢、更不易吸收 P；减小使其更快跟随长期漂移。推荐 30–180 s。

BSLOW_UPDATE_RATE = 1.0
# EWMA 更新倍率。增大让允许更新时跟随更快；减小让 Bslow 更保守。推荐 0.3–1.5。

BSLOW_STABLE_EDGE_THRESHOLD = 3.5
# 允许更新时的候选边沿分数上限。增大让更多边沿附近样本更新；减小会扩大冻结区。推荐 2.5–6.0。

BSLOW_STABLE_SLOPE_THRESHOLD = 3e-4
# 允许更新时 |dBm/dt| 上限（Udeco/s）。增大可在更明显趋势中更新但可能吸收 P；减小更严格冻结。推荐 1e-4–1e-3。

BSLOW_STABLE_OFFSET_THRESHOLD = 1e-3
# Bm 与上一 Bslow 的最大允许偏差（Udeco）。增大可从较大偏差恢复但更易吸收压力平台；减小更保守。推荐 5e-4–5e-3。

J_SMOOTH_WINDOW_SEC = 0.10
# 仅对 Non-contact/Transition 内的模态差信号做短中值平滑（秒）。增大使 J 更平滑；减小保留更陡边沿和更多噪声。推荐 0.05–0.20 s。

# =========================
# 可视化
# =========================
PLOT_START_SEC = None
# 局部观察起点（秒）；None 表示从开头。设大可聚焦后段，不影响算法结果。

PLOT_END_SEC = None
# 局部观察终点（秒）；None 表示到结尾。设小可聚焦前段，不影响算法结果。

NORMALIZE_COMPONENTS_FOR_PLOT = False
# True 时仅对摘要图的绘图副本做 median/MAD 标准化；False 显示真实物理数值。不会修改 result 中任何数组。

## 1. 环境、数据与算法接口

流程保持单线且可解释：`clean → Bpilot → 候选 → 状态机 → Ucontact → Bm_contact → Bslow/P/J`。Bpilot 只服务于检测，最终 Bm 从 Ucontact 独立估计，因此不会继承累计 `clean-J` 的历史偏置。

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display



In [ ]:
from udeco_analysis import load_udeco_sample
from udeco_contact_state_decomposition import ContactStateConfig, decompose_contact_state_udeco

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. 读取唯一指定样本

只需修改第一个代码单元的 `SAMPLE_NAME`。下面会在数据根目录中唯一匹配 `*/<SAMPLE_NAME>_sensor.csv`，并复用仓库的 `load_udeco_sample()` 自动得到 `time / udeco2 / udeco3 / fs`。

In [ ]:
data_root = (Path.cwd() / DATA_ROOT).resolve()
matches = sorted(data_root.glob(f"*/{SAMPLE_NAME}_sensor.csv"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"期望唯一匹配 {SAMPLE_NAME!r}，实际找到 {len(matches)} 个：{matches}"
    )

sensor_csv = matches[0]
sample = load_udeco_sample(
    sensor_csv,
    data_root=data_root,
    fs=int(PROTOCOL_FS_HZ),
)

time = np.asarray(sample.time_s, dtype=float)
udeco2 = np.asarray(sample.udeco2, dtype=float)
udeco3 = np.asarray(sample.udeco3, dtype=float)
fs = float(sample.fs)

print(f"样本：{sample.sample_name}")
print(f"文件：{sensor_csv}")
print(f"采样率：{fs:.1f} Hz；时长：{time[-1]:.2f} s；采样点：{time.size}")
print(f"Udeco2/Udeco3 有限值：{np.isfinite(udeco2).all()} / {np.isfinite(udeco3).all()}")

## 3. 双通道独立分解

两路只共享参数对象，不共享候选、噪声估计、事件列表或状态 mask。`result2` 和 `result3` 均包含完整的中间诊断量，便于定位是候选检测、状态机、Ucontact 重建、Bm 还是 Bslow 出现偏差。

In [ ]:
config = ContactStateConfig(
    hampel_window_s=HAMPEL_WINDOW_SEC,
    hampel_n_sigma=HAMPEL_N_SIGMA,
    pilot_lowess_window_s=PILOT_LOESS_WINDOW_SEC,
    pilot_lowess_robust_iterations=PILOT_LOESS_ROBUST_ITERS,
    candidate_scales=CANDIDATE_SCALES_SEC,
    candidate_z_threshold=CANDIDATE_Z_THRESHOLD,
    candidate_merge_s=CANDIDATE_MERGE_SEC,
    refinement_context_s=REFINEMENT_CONTEXT_SEC,
    candidate_refine_smooth_s=CANDIDATE_REFINE_SMOOTH_SEC,
    detach_peak_slope_z=DETACH_PEAK_SLOPE_Z,
    attach_min_monotonicity=ATTACH_MIN_MONOTONICITY,
    initial_contact_s=INITIAL_CONTACT_SEC,
    state_min_dwell_s=STATE_MIN_DWELL_SEC,
    state_transition_confidence=STATE_TRANSITION_CONFIDENCE,
    contact_anchor_window_s=CONTACT_ANCHOR_WINDOW_SEC,
    max_dynamic_fraction=MAX_DYNAMIC_FRACTION,
    state_pilot_slope_relative_threshold=STATE_PILOT_SLOPE_RELATIVE_THRESHOLD,
    contact_anchor_residual_fraction=CONTACT_ANCHOR_RESIDUAL_FRACTION,
    terminal_min_tail_s=TERMINAL_MIN_TAIL_SEC,
    downstream_min_segment_s=DOWNSTREAM_MIN_SEGMENT_SEC,
    contact_lowess_window_s=CONTACT_LOESS_WINDOW_SEC,
    contact_lowess_robust_iterations=CONTACT_LOESS_ROBUST_ITERS,
    bslow_tau_s=BSLOW_TAU_SEC,
    bslow_update_rate=BSLOW_UPDATE_RATE,
    bslow_stable_edge_threshold=BSLOW_STABLE_EDGE_THRESHOLD,
    bslow_stable_slope_threshold=BSLOW_STABLE_SLOPE_THRESHOLD,
    bslow_stable_offset_threshold=BSLOW_STABLE_OFFSET_THRESHOLD,
    j_smooth_window_s=J_SMOOTH_WINDOW_SEC,
)

result2 = decompose_contact_state_udeco(udeco2, fs, config)
result3 = decompose_contact_state_udeco(udeco3, fs, config)

print(f"Udeco2：候选 {len(result2['transition_candidates'])}，接受事件 {len(result2['J_events'])}")
print(f"Udeco3：候选 {len(result3['transition_candidates'])}，接受事件 {len(result3['J_events'])}")

## 4. 候选与已接受事件表

候选表保留被拒绝的事件及原因，这是调参时最直接的依据；接受事件表只显示真正改变 Contact/Non-contact 状态的 attach/detach 边沿。

In [ ]:
def event_table(events):
    """将事件字典转换为紧凑表格，避免展示冗长的内部字段。"""
    columns = [
        "event_type", "candidate_type", "start_s", "end_s", "duration_s",
        "direction", "delta", "peak_slope_z", "monotonicity", "confidence",
        "accepted", "rejection_reason",
    ]
    frame = pd.DataFrame(events)
    existing = [column for column in columns if column in frame.columns]
    return frame.loc[:, existing] if existing else pd.DataFrame(columns=columns)

print("Udeco2 — 全部候选")
display(event_table(result2["transition_candidates"]))
print("Udeco2 — 状态机接受的 J_events")
display(event_table(result2["J_events"]))

print("Udeco3 — 全部候选")
display(event_table(result3["transition_candidates"]))
print("Udeco3 — 状态机接受的 J_events")
display(event_table(result3["J_events"]))

## 5. 诊断绘图

五行依次回答：尖峰是否清除、Bpilot 是否适合检测、候选为何被接受/拒绝、Ucontact 插值是否合理、最终 Bslow/P/J 是否符合状态约束。绿色背景在最后一行表示 `Bslow update enabled`，即该采样点满足真实稳定 Contact 及三项阈值，Bslow 被允许缓慢更新。

In [ ]:
STATE_COLORS = {
    "Contact": "#dcfce7",
    "Non-contact": "#dbeafe",
    "Transition": "#fed7aa",
    "Uncertain": "#e5e7eb",
}


def _plot_indices(time_axis):
    """只生成绘图索引；时间截取不会改变任何算法数组。"""
    keep = np.ones(time_axis.size, dtype=bool)
    if PLOT_START_SEC is not None:
        keep &= time_axis >= PLOT_START_SEC
    if PLOT_END_SEC is not None:
        keep &= time_axis <= PLOT_END_SEC
    indices = np.flatnonzero(keep)
    if indices.size == 0:
        raise ValueError("绘图时间范围内没有采样点")
    return indices


def _true_runs(mask):
    """把布尔 mask 转成连续闭区间，供状态阴影和更新区绘制。"""
    padded = np.r_[False, np.asarray(mask, dtype=bool), False]
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return zip(changes[::2], changes[1::2] - 1)


def _shade_mask(ax, time_axis, mask, color, label, alpha=0.22):
    """将 mask 的连续区间画成半透明背景；同类状态只进图例一次。"""
    first = True
    for start, end in _true_runs(mask):
        ax.axvspan(
            time_axis[start], time_axis[end], color=color, alpha=alpha,
            label=label if first else None, linewidth=0,
        )
        first = False


def _shade_states(ax, time_axis, result):
    """在状态诊断图中叠加 Contact、Non-contact、Transition 与 Uncertain。"""
    _shade_mask(ax, time_axis, result["contact_mask"], STATE_COLORS["Contact"], "Contact", 0.12)
    _shade_mask(ax, time_axis, result["noncontact_mask"], STATE_COLORS["Non-contact"], "Non-contact", 0.30)
    _shade_mask(ax, time_axis, result["transition_mask"], STATE_COLORS["Transition"], "Transition", 0.28)
    _shade_mask(ax, time_axis, result["uncertain_mask"], STATE_COLORS["Uncertain"], "Uncertain", 0.22)


def _mark_candidates(ax, result):
    """用线型区分候选类型，用透明度区分是否被状态机接受。"""
    shown = set()
    for candidate in result["transition_candidates"]:
        kind = candidate.get("candidate_type", "ambiguous")
        color = {"detach": "#dc2626", "attach": "#2563eb"}.get(kind, "#6b7280")
        style = "-" if candidate.get("accepted", False) else ":"
        alpha = 0.90 if candidate.get("accepted", False) else 0.35
        legend_key = (kind, bool(candidate.get("accepted", False)))
        label = f"{kind} {'accepted' if legend_key[1] else 'candidate'}" if legend_key not in shown else None
        ax.axvline(candidate["start_s"], color=color, linestyle=style, alpha=alpha, linewidth=1.0, label=label)
        shown.add(legend_key)


def plot_diagnostics(time_axis, result_left, result_right, sample_name):
    """绘制 5×2 分层诊断图，两列分别对应 Udeco2 和 Udeco3。"""
    indices = _plot_indices(time_axis)
    t = time_axis[indices]
    results = (("Udeco2", result_left), ("Udeco3", result_right))
    fig, axes = plt.subplots(5, 2, figsize=(17, 18), sharex="col", constrained_layout=True)

    for column, (channel_name, result) in enumerate(results):
        # 1) Raw 与 clean：只应看到孤立尖峰被替换，真实边沿仍被保留。
        ax = axes[0, column]
        ax.plot(t, result["raw"][indices], color="#6b7280", lw=0.8, label="Raw")
        ax.plot(t, result["clean"][indices], color="#111827", lw=1.0, label="clean")
        ax.set_title(f"{channel_name}: Raw / clean")
        ax.legend(loc="best")

        # 2) Bpilot 只用于候选检测；fast residual 放到次坐标避免量纲挤压。
        ax = axes[1, column]
        ax.plot(t, result["clean"][indices], color="#6b7280", lw=0.8, label="clean")
        ax.plot(t, result["Bpilot"][indices], color="#f97316", lw=1.3, label="Bpilot")
        residual_ax = ax.twinx()
        residual_ax.plot(t, result["fast_residual"][indices], color="#7c3aed", lw=0.7, alpha=0.65, label="clean-Bpilot")
        ax.set_title(f"{channel_name}: 检测用 Bpilot 与 fast residual")
        handles1, labels1 = ax.get_legend_handles_labels()
        handles2, labels2 = residual_ax.get_legend_handles_labels()
        ax.legend(handles1 + handles2, labels1 + labels2, loc="best")
        residual_ax.set_ylabel("fast residual")

        # 3) 连续候选分数、阈值、事件线与有限状态机背景。
        ax = axes[2, column]
        ax.plot(t, result["candidate_score"][indices], color="#2563eb", lw=0.9, label="candidate score")
        ax.axhline(CANDIDATE_Z_THRESHOLD, color="#dc2626", ls="--", lw=0.9, label="candidate z limit")
        _shade_states(ax, time_axis, result)
        _mark_candidates(ax, result)
        ax.set_title(f"{channel_name}: attach/detach 候选与 Contact 状态机")
        ax.legend(loc="best", ncol=2)

        # 4) 人工补全区只用于恢复接触模态，再由最终 LOWESS 估计 Bm_contact。
        ax = axes[3, column]
        ax.plot(t, result["Ucontact"][indices], color="#111827", lw=0.9, label="Ucontact")
        ax.plot(t, result["Bm_contact"][indices], color="#f97316", lw=1.4, label="Bm_contact")
        _shade_mask(ax, time_axis, result["ucontact_synthetic_mask"], "#fef3c7", "Ucontact synthetic", 0.35)
        ax.set_title(f"{channel_name}: Ucontact / 最终 Robust LOWESS Bm")
        ax.legend(loc="best")

        # 5) 绿色阴影表示当前点允许 Bslow 更新；空白区表示严格冻结。
        ax = axes[4, column]
        ax.plot(t, result["Bslow"][indices], color="#16a34a", lw=1.3, label="Bslow")
        ax.plot(t, result["P"][indices], color="#dc2626", lw=1.0, label="P")
        ax.plot(t, result["J"][indices], color="#7c3aed", lw=1.0, label="J")
        _shade_mask(ax, time_axis, result["bslow_update_mask"], "#86efac", "Bslow update enabled", 0.20)
        ax.set_title(f"{channel_name}: Bslow / P / J 与更新区")
        ax.set_xlabel("Time (s)")
        ax.legend(loc="best")

        for row in range(5):
            axes[row, column].set_ylabel("Udeco" if row != 2 else "Robust score")
            axes[row, column].set_xlim(t[0], t[-1])

    fig.suptitle(f"{sample_name} — Udeco 接触状态分解诊断", fontsize=16)
    return fig


def _robust_plot_copy(values):
    """仅为绘图创建 robust z-score 副本；不修改算法结果。"""
    copy = np.asarray(values, dtype=float).copy()
    center = np.median(copy)
    scale = 1.4826 * np.median(np.abs(copy - center))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        scale = np.std(copy)
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        scale = 1.0
    return (copy - center) / scale


def plot_summary(time_axis, result_left, result_right, sample_name):
    """绘制最终 2×1 摘要图：每路同时显示 Raw、J、P、Bslow。"""
    indices = _plot_indices(time_axis)
    t = time_axis[indices]
    results = (("Udeco2", result_left), ("Udeco3", result_right))
    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)

    for ax, (channel_name, result) in zip(axes, results):
        series = {
            f"Raw {channel_name}": result["raw"],
            "J(t)": result["J"],
            "P(t)": result["P"],
            "Bslow(t)": result["Bslow"],
        }
        for label, values in series.items():
            plot_values = _robust_plot_copy(values) if NORMALIZE_COMPONENTS_FOR_PLOT else np.asarray(values)
            ax.plot(t, plot_values[indices], lw=1.0, label=label)
        _shade_states(ax, time_axis, result)
        ax.set_title(channel_name)
        ax.set_ylabel("Robust z-score" if NORMALIZE_COMPONENTS_FOR_PLOT else "Udeco")
        ax.set_xlim(t[0], t[-1])
        ax.legend(loc="best", ncol=4)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{sample_name} — Raw / J / P / Bslow 接触状态分解", fontsize=16)
    return fig

In [ ]:
diagnostic_figure = plot_diagnostics(time, result2, result3, SAMPLE_NAME)
plt.show()

## 6. 最终摘要图

最后一格只负责展示当前样本两路的最终结果。状态阴影与诊断图一致；当 `NORMALIZE_COMPONENTS_FOR_PLOT=True` 时，只转换绘图副本，`result2/result3` 中的真实数值保持不变。

In [ ]:
summary_figure = plot_summary(time, result2, result3, SAMPLE_NAME)
plt.show()